# Amazon 用户评论情感分析
逐步复现：数据检查 → 基础模型 → DistilBERT → 独立测试。当前没有执行结果。**第一次只运行第 1–3 步**，核对 ZIP 格式后再继续。

## 1. 环境
基础模型可用 CPU；DistilBERT 建议在 Colab 的运行时设置中选择 GPU。

In [ ]:
%pip install -q "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "torch>=2.4,<3" "transformers>=4.45,<5"

## 2. 挂载 Drive 并选择项目路径
`PROJECT` 是 Google Drive 中项目文件夹的位置，可按实际路径修改。报告和模型会写入该项目的 reports、models 子目录。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/用户情感分析')
TRAIN_ZIP = PROJECT / 'train.zip'
TEST_ZIP = PROJECT / 'test.zip'
assert TRAIN_ZIP.exists() and TEST_ZIP.exists(), '请修改 PROJECT 指向实际项目文件夹'
REPORTS = PROJECT / 'reports'
MODELS = PROJECT / 'models'
REPORTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

## 3. 查看真实文件与标签
不解压或执行 ZIP 中任何代码。预期每个 ZIP 中只有一份 CSV；每行是 label、title、review，label 为 1 或 2。若不匹配，停止并带着输出回到聊天。第一次运行到这里即可。

In [ ]:
import zipfile, csv, io
for path in [TRAIN_ZIP, TEST_ZIP]:
    with zipfile.ZipFile(path) as archive:
        members = [n for n in archive.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/')]
        print(path.name, 'CSV files:', members)
        assert len(members) == 1, '格式不同，请先核对'
        with archive.open(members[0]) as raw:
            reader = csv.reader(io.TextIOWrapper(raw, encoding='utf-8-sig', newline=''))
            for _ in range(5):
                row = next(reader)
                print({'columns': len(row), 'raw_label': row[0], 'text_chars': sum(map(len,row[1:]))})
                assert len(row) == 3 and row[0] in {'1','2'}, '不是预期的 Amazon polarity CSV'
print('检查通过；标签 1=negative，2=positive。后续统一映射为 0/1。')

## 4. 随机抽样与数据质量
流式扫描整个 CSV，仅保留有限样本，避免把 590MB ZIP 全部载入内存。先抽样训练来源 10,000 条、测试来源 2,000 条；这是起步实验，不代表完整数据集成绩。

In [ ]:
"""Read Amazon polarity CSV safely; do not infer labels from row positions."""
import csv
import io
import random
import zipfile
from collections import Counter


def read_amazon_zip(path, sample_size=10000, seed=42):
    """Uniform reservoir sample over a headerless label,title,review CSV."""
    if sample_size < 2:
        raise ValueError('sample_size must be at least 2')
    rng = random.Random(seed)
    sample = []
    counts = Counter()
    with zipfile.ZipFile(path) as archive:
        members = [n for n in archive.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/')]
        if len(members) != 1:
            raise ValueError(f'Expected exactly one CSV, found {members}')
        with archive.open(members[0]) as raw:
            reader = csv.reader(io.TextIOWrapper(raw, encoding='utf-8-sig', newline=''))
            for index, row in enumerate(reader):
                if len(row) != 3 or row[0] not in {'1', '2'}:
                    raise ValueError(f'Invalid Amazon CSV row {index + 1}; expected label 1/2,title,review')
                label = int(row[0]) - 1
                text = (row[1] + ' ' + row[2]).strip()
                if not text:
                    raise ValueError(f'Empty review at row {index + 1}')
                counts[label] += 1
                item = {'text': text, 'label': label}
                if len(sample) < sample_size:
                    sample.append(item)
                else:
                    slot = rng.randrange(index + 1)
                    if slot < sample_size:
                        sample[slot] = item
    if set(counts) != {0, 1}:
        raise ValueError(f'Both classes required; observed {dict(counts)}')
    return sample, {'source_rows': sum(counts.values()), 'source_labels': dict(counts), 'csv_member': members[0], 'seed': seed}


def clean_text(text):
    import html
    import re
    return re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', html.unescape(text))).strip()


In [ ]:
import pandas as pd, json
SEED = 42
rows, train_source = read_amazon_zip(TRAIN_ZIP, sample_size=10000, seed=SEED)
test_rows, test_source = read_amazon_zip(TEST_ZIP, sample_size=2000, seed=SEED)
def prepare(rows):
    frame = pd.DataFrame(rows)
    frame['text'] = frame['text'].map(clean_text)
    frame['key'] = frame['text'].str.casefold()
    conflicts = frame.groupby('key')['label'].nunique()
    bad_keys = set(conflicts[conflicts > 1].index)
    return frame[~frame['key'].isin(bad_keys) & frame['text'].ne('')].drop_duplicates('key').reset_index(drop=True)
pool = prepare(rows)
heldout = prepare(test_rows)
overlap = heldout['key'].isin(set(pool['key']))
print('测试来源与训练来源重复条数（将移除）:', int(overlap.sum()))
heldout = heldout[~overlap].reset_index(drop=True)
from sklearn.model_selection import train_test_split
train, valid = train_test_split(pool, test_size=0.2, random_state=SEED, stratify=pool['label'])
assert not set(train['key']) & set(valid['key'])
assert not set(pool['key']) & set(heldout['key'])
for name, frame in [('train',train),('valid',valid),('heldout',heldout)]:
    assert set(frame['label']) == {0,1}, f'{name} 必须包含两类'
    print(name, len(frame), frame['label'].value_counts().to_dict())
    print('评论长度分位数:', frame['text'].str.len().quantile([.5,.9,.99]).to_dict())
manifest = {'train_source':train_source,'test_source':test_source,'train_rows':len(train),'valid_rows':len(valid),'test_rows':len(heldout),'removed_test_overlap':int(overlap.sum())}
(REPORTS/'data_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

## 5. TF-IDF + Logistic Regression
Pipeline 保证 TF-IDF 只拟合训练数据。验证集用于判断模型与设置；测试集暂不查看成绩。

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
def metrics(y,p):
    return {'accuracy':float(accuracy_score(y,p)), 'macro_f1':float(f1_score(y,p,average='macro')), 'confusion_matrix':confusion_matrix(y,p,labels=[0,1]).tolist()}
baseline = make_pipeline(TfidfVectorizer(ngram_range=(1,2),max_features=50000), LogisticRegression(max_iter=1000,random_state=SEED))
baseline.fit(train['text'],train['label'])
print('Baseline validation:', metrics(valid['label'],baseline.predict(valid['text'])))

## 6. DistilBERT 微调
固定种子，tokenizer 自动添加特殊 token 和 attention mask。每轮只评估验证集，按 Macro-F1 保存最优 checkpoint。本格只定义训练需要的组件。

In [ ]:
import torch, random, numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
MODEL_NAME = 'distilbert/distilbert-base-uncased'
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
class Reviews(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        item = tokenizer(row['text'], truncation=True,padding='max_length',max_length=MAX_LENGTH,return_tensors='pt')
        item = {k:v.squeeze(0) for k,v in item.items()}
        item['labels'] = torch.tensor(int(row['label']), dtype=torch.long)
        return item
def loader(frame, shuffle=False):
    return DataLoader(Reviews(frame),batch_size=BATCH_SIZE,shuffle=shuffle,generator=torch.Generator().manual_seed(SEED))
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2,id2label={0:'NEGATIVE',1:'POSITIVE'},label2id={'NEGATIVE':0,'POSITIVE':1}).to(device)
def predict_frame(model, frame):
    model.eval(); predictions = []
    with torch.no_grad():
        for batch in loader(frame):
            batch = {k:v.to(device) for k,v in batch.items()}
            predictions.extend(model(**batch).logits.argmax(-1).cpu().tolist())
    return predictions

### 执行训练
GPU 的资源可用性取决于你的 Colab 会话。没有 GPU 时，先完成基础模型即可，不必在 CPU 上等完整微调。

In [ ]:
assert device.type == 'cuda', '建议切换到 GPU 后再运行此训练单元'
optimizer = torch.optim.AdamW(model.parameters(),lr=2e-5)
train_loader = loader(train,shuffle=True)
best_f1 = -1
history = []
for epoch in range(EPOCHS):
    model.train(); total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        batch = {k:v.to(device) for k,v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        total_loss += loss.item()
    result = metrics(valid['label'],predict_frame(model,valid))
    history.append({'epoch':epoch+1,'train_loss':total_loss/len(train_loader),**result})
    print(history[-1])
    if result['macro_f1'] > best_f1:
        best_f1 = result['macro_f1']
        model.save_pretrained(MODELS/'distilbert')
        tokenizer.save_pretrained(MODELS/'distilbert')
(REPORTS/'training_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')

## 7. 最终评估与报告
确定实验设置后才运行这里。两模型使用相同独立测试样本。测试成绩一旦用于改模型，该测试集就不再是全新的最终评估集。错误 CSV 留在 Drive，公开前检查样例内容。

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODELS/'distilbert').to(device)
baseline_pred = baseline.predict(heldout['text'])
bert_pred = predict_frame(model,heldout)
results = {'dataset':manifest,'config':{'model':MODEL_NAME,'max_length':MAX_LENGTH,'batch_size':BATCH_SIZE,'epochs':EPOCHS,'seed':SEED},'models':{}}
for name,pred in [('tfidf_logreg',baseline_pred),('distilbert',bert_pred)]:
    result = metrics(heldout['label'],pred)
    results['models'][name] = result
    print(name,result)
    ConfusionMatrixDisplay(np.array(result['confusion_matrix']),display_labels=['negative','positive']).plot()
    plt.title(name); plt.tight_layout(); plt.savefig(REPORTS/f'{name}_confusion.png',dpi=160); plt.show(); plt.close()
    errors = heldout.assign(prediction=pred)
    errors[errors['label'] != errors['prediction']][['text','label','prediction']].to_csv(REPORTS/f'errors_{name}.csv',index=False)
(REPORTS/'metrics.json').write_text(json.dumps(results,indent=2),encoding='utf-8')
import subprocess,sys
(REPORTS/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
print('报告目录:', REPORTS)

## 8. 单条预测与作品集
模型适用范围为英文评论正负面二分类。Softmax 分数不是经过校准的可信度。随后人工阅读至少 20 条误判，填写 README 的真实成绩与局限。

In [ ]:
text = 'The packaging was nice, but the product stopped working after one day.'
model.eval()
with torch.no_grad():
    inputs = tokenizer(text,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(device)
    scores = model(**inputs).logits.softmax(-1)[0]
print({'text':text,'label':model.config.id2label[int(scores.argmax())],'softmax_score':float(scores.max())})